# 03 - 指令集与汇编基础（Instruction Set & Assembly）

这个 notebook 让你用 **Python 写一个汇编器（Assembler）**，亲手把汇编语言翻译成机器码，再运行它。你将：
- 理解指令编码格式
- 自己写汇编、自己汇编、自己跑
- 对比 RISC 和 CISC 的差异

## 怎么用？
从上到下依次运行每个 cell（`Shift + Enter`），观察输出，动手改参数实验。

## 今日学习路线

| # | 内容 |
|---|------|
| 3.1 | 机器码是怎么来的 |
| 3.2 | 写一个汇编器 |
| 3.3 | 寻址模式实践 |
| 3.4 | RISC vs CISC 对比 |
| 3.5 | 函数调用约定 |
| 3.6 | 从高级语言到汇编 |

---
## 3.1 从汇编到机器码

### 编译链

```
C/Java/Python → 编译器 → 汇编语言 → 汇编器(Assembler) → 机器码(二进制)
```

汇编器的工作：把人类可读的助记符（`ADD R0, R1, R2`）翻译成 CPU 能执行的二进制（`1110 00 0 0100 1 0001 0 000 0000 0010`）。

In [ ]:
# ============================================
# 一个最小汇编器
# 输入: 汇编代码 (字符串)
# 输出: 机器码 (整数数组)
# ============================================

class Assembler:
    """
    支持一个简单 RISC ISA 的汇编器
    
    指令格式 (16-bit 定长):
    ┌──────┬────┬────┬────┬──────┐
    │opcode│ Rd │ Rs1│ Rs2│ imm6 │
    │ 4bit │3bit│3bit│3bit│ 3bit │
    └──────┴────┴────┴────┴──────┘
    
    但 LOAD/STORE 使用不同格式:
    ┌──────┬────┬────┬───────────┐
    │opcode│ Rd │ Rs1│  imm8     │
    │ 4bit │3bit│3bit│  8bit     │
    └──────┴────┴────┴───────────┘
    
    为简单起见，我们编码为「操作码 + 操作数」的列表形式。
    """
    
    # 操作码分配
    OPCODES = {
        'HALT': 0x00,
        'LOAD': 0x01,
        'STORE':0x02,
        'ADD':  0x03,
        'SUB':  0x04,
        'MUL':  0x05,
        'JMP':  0x06,
        'JZ':   0x07,
        'CMP':  0x08,
        'MOV':  0x09,
        'INC':  0x0A,
        'AND':  0x0B,
        'OR':   0x0C,
        'XOR':  0x0D,
    }
    
    REVERSE_OPCODES = {v: k for k, v in OPCODES.items()}
    
    def __init__(self):
        self.machine_code = []  # 输出的机器码（整数列表）
        self.labels = {}         # 标签 → 地址
        self.pending_labels = {} # 待回填的标签引用
    
    def parse_register(self, token):
        """解析 'R0' → 0, 'R1' → 1"""
        if token.startswith('R') or token.startswith('r'):
            return int(token[1:])
        return None
    
    def parse_immediate(self, token):
        """解析 '#42' → 42, '#0x1A' → 26"""
        if token.startswith('#'):
            val = token[1:]
            if val.startswith('0x') or val.startswith('0X'):
                return int(val, 16)
            return int(val)
        return None
    
    def assemble_line(self, line, current_addr):
        """汇编一行代码"""
        # 去除注释
        if ';' in line:
            line = line[:line.index(';')]
        line = line.strip()
        
        if not line:
            return []
        
        # 标签？
        if line.endswith(':'):
            label = line[:-1]
            self.labels[label] = current_addr
            # 回填之前引用这个标签的地方
            if label in self.pending_labels:
                for patch_addr, offset in self.pending_labels[label]:
                    self.machine_code[patch_addr + offset] = current_addr
            return []
        
        parts = line.replace(',', ' ').split()
        if not parts:
            return []
        
        mnemonic = parts[0].upper()
        
        # ── 各种指令的编码 ──
        
        if mnemonic == 'HALT':
            return [self.OPCODES['HALT']]
        
        elif mnemonic == 'INC':
            rx = self.parse_register(parts[1])
            return [self.OPCODES['INC'], rx]
        
        elif mnemonic == 'MOV':
            # MOV Rd, Rs
            rd = self.parse_register(parts[1])
            rs = self.parse_register(parts[2])
            return [self.OPCODES['MOV'], rd, rs]
        
        elif mnemonic in ['ADD', 'SUB', 'MUL', 'AND', 'OR', 'XOR']:
            # OP Rd, Rs1, Rs2
            rd = self.parse_register(parts[1])
            rs1 = self.parse_register(parts[2])
            rs2 = self.parse_register(parts[3])
            return [self.OPCODES[mnemonic], rd, rs1, rs2]
        
        elif mnemonic == 'LOAD':
            # LOAD Rd, [addr]  or  LOAD Rd, [Rs]
            rd = self.parse_register(parts[1])
            # 取 [addr] 部分
            addr_part = parts[2]
            if addr_part.startswith('R') or addr_part.startswith('r'):
                # 间接寻址: LOAD Rd, Rs  (简化：Rs 中存在地址)
                rs = self.parse_register(addr_part)
                return [self.OPCODES['LOAD'], rd, rs]
            else:
                # 直接地址
                addr = int(addr_part, 0)
                return [self.OPCODES['LOAD'], rd, addr]
        
        elif mnemonic == 'STORE':
            # STORE [addr], Rs
            addr = int(parts[1], 0)
            rs = self.parse_register(parts[2])
            return [self.OPCODES['STORE'], addr, rs]
        
        elif mnemonic == 'CMP':
            rs1 = self.parse_register(parts[1])
            rs2 = self.parse_register(parts[2])
            return [self.OPCODES['CMP'], rs1, rs2]
        
        elif mnemonic == 'JMP':
            target = parts[1]
            if target.isdigit() or target.startswith('0x'):
                addr = int(target, 0)
                return [self.OPCODES['JMP'], addr]
            else:
                # 标签引用
                if target in self.labels:
                    return [self.OPCODES['JMP'], self.labels[target]]
                else:
                    self.pending_labels.setdefault(target, []).append(
                        (len(self.machine_code), 1))
                    return [self.OPCODES['JMP'], 0xFFFF]  # 占位
        
        elif mnemonic == 'JZ':
            target = parts[1]
            if target.isdigit() or target.startswith('0x'):
                addr = int(target, 0)
                return [self.OPCODES['JZ'], addr]
            else:
                if target in self.labels:
                    return [self.OPCODES['JZ'], self.labels[target]]
                else:
                    self.pending_labels.setdefault(target, []).append(
                        (len(self.machine_code), 1))
                    return [self.OPCODES['JZ'], 0xFFFF]  # 占位
        
        else:
            print(f"⚠️ 未知指令: {line}")
            return []
    
    def assemble(self, code):
        """汇编完整程序"""
        lines = code.strip().split('\n')
        
        # 第一遍：确定地址和标签
        addr = 0
        expanded_lines = []
        for line in lines:
            cleaned = line.split(';')[0].strip()
            if not cleaned:
                continue
            if cleaned.endswith(':'):
                self.labels[cleaned[:-1]] = addr
                expanded_lines.append(cleaned)
                continue
            expanded_lines.append(cleaned)
            # 算这条指令占多少字
            parts = cleaned.replace(',', ' ').split()
            mnemonic = parts[0].upper()
            if mnemonic in ['ADD', 'SUB', 'MUL', 'AND', 'OR', 'XOR']:
                addr += 4
            elif mnemonic in ['LOAD', 'STORE', 'JMP', 'JZ']:
                addr += 2
            elif mnemonic in ['CMP', 'MOV']:
                addr += 3  # MOV: 3, CMP: 3
            elif mnemonic == 'INC':
                addr += 2
            elif mnemonic == 'HALT':
                addr += 1
        
        # 第二遍：生成机器码
        self.machine_code = []
        for line in expanded_lines:
            current_addr = len(self.machine_code)
            codes = self.assemble_line(line, current_addr)
            self.machine_code.extend(codes)
        
        return self.machine_code
    
    def disassemble(self, machine_code):
        """反汇编：机器码 → 汇编"""
        lines = []
        i = 0
        while i < len(machine_code):
            opcode = machine_code[i]
            mnemonic = self.REVERSE_OPCODES.get(opcode, f'???({opcode})')
            addr = f"[{i:3d}]"
            
            if opcode == 0x00:  # HALT
                lines.append(f"{addr} HALT")
                i += 1
            elif opcode == 0x01:  # LOAD
                lines.append(f"{addr} LOAD R{machine_code[i+1]}, [0x{machine_code[i+2]:02X}]")
                i += 3
            elif opcode == 0x02:  # STORE
                lines.append(f"{addr} STORE [0x{machine_code[i+1]:02X}], R{machine_code[i+2]}")
                i += 3
            elif opcode in [0x03, 0x04, 0x05, 0x0B, 0x0C, 0x0D]:  # ADD/SUB/MUL/AND/OR/XOR
                lines.append(f"{addr} {mnemonic} R{machine_code[i+1]}, R{machine_code[i+2]}, R{machine_code[i+3]}")
                i += 4
            elif opcode == 0x06:  # JMP
                lines.append(f"{addr} JMP 0x{machine_code[i+1]:04X}")
                i += 2
            elif opcode == 0x07:  # JZ
                lines.append(f"{addr} JZ 0x{machine_code[i+1]:04X}")
                i += 2
            elif opcode == 0x08:  # CMP
                lines.append(f"{addr} CMP R{machine_code[i+1]}, R{machine_code[i+2]}")
                i += 3
            elif opcode == 0x09:  # MOV
                lines.append(f"{addr} MOV R{machine_code[i+1]}, R{machine_code[i+2]}")
                i += 3
            elif opcode == 0x0A:  # INC
                lines.append(f"{addr} INC R{machine_code[i+1]}")
                i += 2
            else:
                lines.append(f"{addr} ??? {opcode}")
                i += 1
        
        return '\n'.join(lines)

print("✅ 汇编器已定义")
print("   用法: asm = Assembler(); machine_code = asm.assemble('你的汇编代码')")

In [ ]:
# ============================================
# 测试：写汇编、汇编、查看机器码
# ============================================

asm = Assembler()

my_code = """
    ; 计算 (100 + 50) * 3
    LOAD R0, [0x90]    ; R0 = 100
    LOAD R1, [0x91]    ; R1 = 50
    ADD  R2, R0, R1    ; R2 = R0 + R1
    LOAD R3, [0x92]    ; R3 = 3
    MUL  R0, R2, R3    ; R0 = R2 * R3
    STORE [0x80], R0   ; mem[0x80] = R0
    HALT
"""

machine_code = asm.assemble(my_code)
print("📋 机器码（整数列表）:")
print(f"   {machine_code}\n")
print("📋 反汇编:")
print(asm.disassemble(machine_code))

### 把前面写的 CPU 接入汇编器

现在我们可以**写汇编 → 汇编成机器码 → 加载到 CPU → 执行**，一条龙！

In [ ]:
# 复用上一章的 CPU
# 如果不在同一个 notebook，重新定义

class Memory:
    def __init__(self, size=256):
        self.size = size
        self.cells = [0] * size
    def read(self, addr):
        return self.cells[addr] if 0 <= addr < self.size else 0
    def write(self, addr, val):
        if 0 <= addr < self.size:
            self.cells[addr] = val & 0xFFFF

class Registers:
    def __init__(self):
        self.PC = 0; self.IR = 0; self.MAR = 0; self.MBR = 0
        self.ACC = 0; self.SP = 0xFF
        self.R = [0, 0, 0, 0]
        self.Z = False; self.N = False; self.C = False; self.V = False
    def update_flags(self, result):
        self.Z = (result == 0); self.N = (result < 0)

class ALU:
    ADD=0; SUB=1; MUL=2; AND=3; OR=4; XOR=5
    def compute(self, opcode, a, b=0):
        if opcode == 0: return a+b, {'Z':a+b==0, 'N':(a+b)<0, 'C':False, 'V':False}
        if opcode == 1: return a-b, {'Z':a-b==0, 'N':(a-b)<0, 'C':False, 'V':False}
        if opcode == 2: return a*b, {'Z':a*b==0, 'N':(a*b)<0, 'C':False, 'V':False}
        if opcode == 3: return a&b, {'Z':a&b==0, 'N':False, 'C':False, 'V':False}
        if opcode == 4: return a|b, {'Z':a|b==0, 'N':False, 'C':False, 'V':False}
        if opcode == 5: return a^b, {'Z':a^b==0, 'N':False, 'C':False, 'V':False}
        return 0, {}

class CPU:
    HALT=0x00; LOAD=0x01; STORE=0x02; ADD=0x03; SUB=0x04; MUL=0x05
    JMP=0x06; JZ=0x07; CMP=0x08; MOV=0x09; INC=0x0A; AND=0x0B; OR=0x0C; XOR=0x0D
    MNEMONICS = {0x00:'HALT',0x01:'LOAD',0x02:'STORE',0x03:'ADD',0x04:'SUB',
                 0x05:'MUL',0x06:'JMP',0x07:'JZ',0x08:'CMP',0x09:'MOV',
                 0x0A:'INC',0x0B:'AND',0x0C:'OR',0x0D:'XOR'}
    
    def __init__(self):
        self.mem = Memory(256); self.reg = Registers(); self.alu = ALU()
        self.running = False; self.cycle = 0
    def _read_mem(self, addr):
        self.reg.MAR = addr; self.reg.MBR = self.mem.read(addr); return self.reg.MBR
    def _write_mem(self, addr, val):
        self.reg.MAR = addr; self.reg.MBR = val; self.mem.write(addr, val)
    def fetch(self):
        self.reg.IR = self._read_mem(self.reg.PC); self.reg.PC += 1; return self.reg.IR
    def _update_flags_dict(self, d):
        self.reg.Z=d.get('Z',False); self.reg.N=d.get('N',False); self.reg.C=d.get('C',False); self.reg.V=d.get('V',False)
    
    def decode_execute(self, opcode):
        if opcode == self.HALT: self.running = False; return "HALT"
        elif opcode == self.LOAD:
            rd, addr = self.fetch(), self.fetch()
            val = self._read_mem(addr); self.reg.R[rd] = val; self.reg.update_flags(val)
            return f"LOAD R{rd}, [{addr}] → R{rd}={val}"
        elif opcode == self.STORE:
            addr, rs = self.fetch(), self.fetch()
            val = self.reg.R[rs]; self._write_mem(addr, val)
            return f"STORE [{addr}], R{rs} → mem[{addr}]={val}"
        elif opcode in [self.ADD, self.SUB, self.MUL, self.AND, self.OR, self.XOR]:
            rd, rs1, rs2 = self.fetch(), self.fetch(), self.fetch()
            a, b = self.reg.R[rs1], self.reg.R[rs2]
            result, flags = self.alu.compute(opcode - self.ADD, a, b)
            self.reg.R[rd] = result; self._update_flags_dict(flags)
            return f"{self.MNEMONICS[opcode]} R{rd},R{rs1},R{rs2} → {a} op {b} = {result}"
        elif opcode == self.JMP:
            addr = self.fetch(); self.reg.PC = addr
            return f"JMP → PC={addr}"
        elif opcode == self.JZ:
            addr = self.fetch()
            if self.reg.Z: self.reg.PC = addr; return f"JZ → 跳转! PC={addr}"
            return f"JZ → 不跳转 (Z=0)"
        elif opcode == self.CMP:
            rs1, rs2 = self.fetch(), self.fetch()
            a, b = self.reg.R[rs1], self.reg.R[rs2]; _, flags = self.alu.compute(1, a, b)
            self._update_flags_dict(flags)
            return f"CMP R{rs1},R{rs2} ({a} vs {b})"
        elif opcode == self.MOV:
            rd, rs = self.fetch(), self.fetch()
            self.reg.R[rd] = self.reg.R[rs]; return f"MOV R{rd},R{rs}"
        elif opcode == self.INC:
            rx = self.fetch(); self.reg.R[rx] += 1; self.reg.update_flags(self.reg.R[rx])
            return f"INC R{rx} → {self.reg.R[rx]}"
        return f"???"
    
    def load_machine_code(self, machine_code, start_addr=0):
        for i, val in enumerate(machine_code):
            self.mem.write(start_addr + i, val)
    
    def run(self):
        self.reg.PC = 0; self.cycle = 0; self.running = True
        print("╔" + "═" * 50 + "╗")
        print("║  🚀 CPU 执行汇编程序" + " " * 29 + "║")
        print("╚" + "═" * 50 + "╝\n")
        while self.running and self.reg.PC < self.mem.size:
            self.cycle += 1; opcode = self.fetch()
            desc = self.decode_execute(opcode)
            mnemonic = self.MNEMONICS.get(opcode, '???')
            print(f"  [{self.cycle:2d}] {mnemonic:<5s} {desc}")
            if self.cycle > 100: break
        print(f"\n✅ 执行完毕，{self.cycle} 个周期")

print("✅ CPU 已就绪")

In [ ]:
# ============================================
# 汇编 → 执行 一条龙
# ============================================

# 1. 写汇编代码
asm = Assembler()
assembly_code = """
    ; 计算 1+2+3+4+5 = 15
    LOAD R0, [0xA0]    ; R0 = 0 (累加和)
    LOAD R1, [0xA1]    ; R1 = 5 (计数器)
    LOAD R2, [0xA2]    ; R2 = 1 (常量1)
    LOAD R3, [0xA3]    ; R3 = 0 (常量0)
loop:
    ADD  R0, R0, R1    ; sum += counter
    SUB  R1, R1, R2    ; counter--
    CMP  R1, R3        ; counter == 0?
    JZ   done          ; 如果是，结束
    JMP  loop          ; 否则继续
done:
    STORE [0x80], R0   ; 存结果
    HALT
"""

# 2. 汇编
machine_code = asm.assemble(assembly_code)
print("📋 标签表:", asm.labels)
print(f"📋 机器码: {machine_code}\n")

# 3. 加载到 CPU 并执行
cpu = CPU()
cpu.load_machine_code(machine_code)

# 手动设置数据
cpu.mem.write(0xA0, 0)   # sum = 0
cpu.mem.write(0xA1, 5)   # counter = 5
cpu.mem.write(0xA2, 1)   # constant 1
cpu.mem.write(0xA3, 0)   # constant 0

cpu.run()
print(f"\n🔍 验证: mem[0x80] = {cpu.mem.read(0x80)}  (预期: 15)")

### ✏️ 自己动手 — 用汇编写阶乘

写一段汇编程序，计算 5! = 5×4×3×2×1 = 120。

In [ ]:
# 👇 在这里写你的汇编程序
asm2 = Assembler()

factorial_code = """
    ; 计算 5! = 120
    ; 提示：
    ;   R0 = product (初始 = 1)
    ;   R1 = counter (初始 = 5)
    ;   R2 = 1 (用于递减)
    ;   R3 = 0 (用于比较)
    ;
    ;   循环: product = product * counter
    ;         counter = counter - 1
    ;         如果 counter > 0, 继续循环
    ;         (CMP counter, 0 → JZ 跳出)
    ;
    ; 你的代码：
    
    
"""

# machine_code = asm2.assemble(factorial_code)
# cpu2 = CPU()
# cpu2.load_machine_code(machine_code)
# cpu2.mem.write(0xA0, 1)   # product = 1
# cpu2.mem.write(0xA1, 5)   # counter = 5
# cpu2.mem.write(0xA2, 1)   # constant 1
# cpu2.mem.write(0xA3, 0)   # constant 0
# cpu2.run()
# print(f"\n🔍 mem[0x80] = {cpu2.mem.read(0x80)}  (预期: 120)")
print("✏️ 写完后取消注释来运行")

---
## 3.2 寻址模式实战

寻址模式 = 指令如何找到它的操作数。我们用代码逐一演示。

In [ ]:
# ============================================
# 各种寻址模式演示
# ============================================

def addressing_mode_demo():
    print("📖 寻址模式演示\n")
    
    # 准备内存
    mem = Memory(256)
    mem.write(100, 42)     # 地址 100 处存了 42
    mem.write(200, 100)    # 地址 200 处存了地址 100
    mem.write(108, 99)     # 地址 108 处存了 99
    
    print("内存快照:")
    print(f"  mem[100] = 42")
    print(f"  mem[108] = 99")
    print(f"  mem[200] = 100 (指向 mem[100])\n")
    
    # 1. 立即数寻址 — 值就在指令里
    print("1️⃣ 立即数寻址 (Immediate):")
    print("   指令: ADD R0, #42")
    print("   → 操作数 42 直接在指令中\n")
    
    # 2. 寄存器直接寻址 — 值在寄存器里
    print("2️⃣ 寄存器直接寻址 (Register Direct):")
    print("   指令: ADD R0, R1")
    print("   → 操作数是 R1 的值\n")
    
    # 3. 直接寻址 — 地址在指令中
    print("3️⃣ 直接寻址 (Direct):")
    addr = 100
    val = mem.read(addr)
    print(f"   指令: LOAD R0, [{addr}]")
    print(f"   → R0 = mem[{addr}] = {val}\n")
    
    # 4. 间接寻址 — 寄存器里有地址，去那个地址取值
    print("4️⃣ 间接寻址 (Indirect):")
    r1 = 200  # R1 存着地址 200
    ptr_addr = mem.read(r1)  # mem[200] = 100
    val = mem.read(ptr_addr)  # mem[100] = 42
    print(f"   指令: LOAD R0, [R1]  (R1 = {r1})")
    print(f"   → addr = mem[R1] = mem[{r1}] = {ptr_addr}")
    print(f"   → R0 = mem[{ptr_addr}] = {val}\n")
    
    # 5. 基址+偏移 — 寄存器值 + 偏移 = 地址
    print("5️⃣ 基址+偏移寻址 (Base+Offset):")
    base = 100
    offset = 8
    addr = base + offset
    val = mem.read(addr)
    print(f"   指令: LOAD R0, [R1, #{offset}]  (R1 = {base})")
    print(f"   → addr = R1 + {offset} = {addr}")
    print(f"   → R0 = mem[{addr}] = {val}")
    print(f"   💡 常见用途: 结构体字段 (base=结构体首地址, offset=字段偏移)\n")

addressing_mode_demo()

---
## 3.3 RISC vs CISC 对比

同一个计算任务，两种风格的指令集写出来截然不同。

In [ ]:
# ============================================
# 同一个任务：x = a + b  用两种 ISA 写
# ============================================

print("📊 任务: C = A + B，A、B、C 都在内存中\n")

print("┌─ RISC 风格 (ARM-like) ─────────────────────┐")
risc_code = """
    LOAD R0, [A]    ; 第1步：从内存加载 A
    LOAD R1, [B]    ; 第2步：从内存加载 B
    ADD  R2, R1, R0 ; 第3步：寄存器相加
    STORE [C], R2   ; 第4步：存回内存
"""
for line in risc_code.strip().split('\n'):
    print(f"  {line}")
print("  → 4 条指令，每条 4 字节 = 16 字节")
print("  → 只有 LOAD/STORE 访问内存")
print("└─────────────────────────────────────────────┘\n")

print("┌─ CISC 风格 (x86-like) ─────────────────────┐")
print("  MOV  AX, [A]   ; 从内存加载 A")
print("  ADD  AX, [B]   ; 直接加内存中的 B")
print("  MOV  [C], AX   ; 存回内存")
print("  → 3 条指令，但长度不等（~10 字节）")
print("  → ADD 指令可以直接访问内存")
print("└─────────────────────────────────────────────┘\n")

print("💡 关键区别:")
print("  - RISC: 指令多但每条简单、定长 → 易于流水线")
print("  - CISC: 指令少但每条复杂、变长 → 译码复杂")
print("  - 现代 x86 内部：把 CISC 拆成 RISC 微操作执行")

---
## 3.4 函数调用约定

当函数 A 调用函数 B 时，参数怎么传、返回值怎么给、怎么返回——这些规则叫**调用约定（Calling Convention）**。

In [ ]:
# ============================================
# 模拟函数调用过程
# ============================================

class StackMemory:
    """带栈支持的内存"""
    def __init__(self, size=256):
        self.cells = [0] * size
        self.SP = 0xF0  # 栈从高端往低端长
    
    def push(self, val):
        """入栈：SP 减小，值写入新位置"""
        self.SP -= 1
        self.cells[self.SP] = val
        return self.SP
    
    def pop(self):
        """出栈：读值，SP 增大"""
        val = self.cells[self.SP]
        self.SP += 1
        return val
    
    def dump_stack(self, n=8):
        """显示栈区内容"""
        print(f"  栈指针 SP = 0x{self.SP:02X}\n")
        for i in range(n):
            addr = self.SP + i
            if addr < len(self.cells):
                marker = ' ← SP' if addr == self.SP else ''
                print(f"  [0x{addr:02X}] = {self.cells[addr]:5d}{marker}")

def simulate_function_call():
    """
    模拟:
    main() 调用 add(x=5, y=3)，add 返回 x+y = 8
    
    调用约定 (简化 ARM):
    - 参数: R0=x, R1=y
    - 返回值: R0
    - 返回地址: LR (Link Register)
    - 被调用者保存的寄存器: R4-R11
    """
    
    mem = StackMemory()
    
    # 寄存器模拟
    R = [0] * 16  # R0-R15
    R[0] = 5       # 参数 x
    R[1] = 3       # 参数 y
    R[14]= 0x0100  # LR = main 的返回地址（main 执行完后回 OS）
    
    print("=" * 55)
    print("📞 main() 调用 add(x=5, y=3)")
    print("=" * 55)
    
    # ── 步骤 1: main 保存 LR 到栈（因为 main 也要调用别人）──
    print("\n① main: PUSH LR (保存返回地址到栈)")
    mem.push(R[14])
    mem.dump_stack(4)
    
    # ── 步骤 2: main 设置参数，跳转到 add ──
    print("\n② main: BL add  (R0=5, R1=3, LR=返回地址)")
    R[14]= 0x0020  # LR = add 返回后的下一条地址
    print(f"   参数: R0={R[0]}, R1={R[1]}")
    print(f"   LR = 0x{R[14]:04X} (add 返回后会跳到这里)")
    
    # ── 步骤 3: add 函数的开头，保存现场 ──
    print("\n③ add 开头: PUSH {R4, LR} (保存调用者可能用的寄存器)")
    mem.push(R[4])   # 保存可能被覆盖的 R4
    mem.push(R[14])  # 保存 LR（add 内部可能调其他函数）
    mem.dump_stack(4)
    
    # ── 步骤 4: add 函数体 ──
    print("\n④ add 函数体: R0 = R0 + R1")
    R[0] = R[0] + R[1]  # 和存到 R0（也是返回值）
    print(f"   R0 = 5 + 3 = {R[0]}")
    
    # ── 步骤 5: add 返回 —— 恢复现场，跳回 ──
    print("\n⑤ add 返回: POP {R4, PC} (恢复 R4，LR→PC 即返回)")
    return_addr = mem.pop()  # 弹出 LR
    mem.pop()  # 弹出旧的 R4
    print(f"   PC ← 0x{return_addr:04X} (返回 main)")
    
    # ── 步骤 6: main 收尾 ──
    print("\n⑥ main: POP LR (恢复自己的返回地址)")
    mem.pop()
    print(f"   R0 = {R[0]} (add 的返回值)")
    
    print(f"\n✅ 函数调用完毕: add(5, 3) = {R[0]}")

simulate_function_call()

### 调用约定的关键规则总结

```
        调用方 (Caller)                       被调用方 (Callee)
        ─────────────                        ────────────────
  ① 把参数放到 R0~R3                 ③ PUSH 需要用到的寄存器
  ② BL func  (LR ← PC+1)            ④ 执行函数体
     ...                             ⑤ 返回值放在 R0~R1
  ⑥ 读取返回值 R0                    ⑥ POP 恢复寄存器 + 返回
```

---
## 3.5 从高级语言到汇编

看看你最熟悉的 Python/C 语句，对应什么汇编。

In [ ]:
# ============================================
# Python → 伪汇编 映射表
# ============================================

print("📊 高级语言 → 汇编 对照表\n")

examples = [
    ("Python", "C", "伪汇编 (ARM风格)", "说明"),
    ("-"*20, "-"*20, "-"*20, "-"*20),
    ("x = 5", "int x = 5;", "MOV R0, #5  ; R0=5", "立即数赋值"),
    ("x = y", "x = y;", "MOV R0, R1  ; R0=R1", "寄存器拷贝"),
    ("x = y + z", "x = y + z;", "ADD R0, R1, R2", "加法"),
    ("x = *ptr", "x = *ptr;", "LDR R0, [R1]", "指针解引用"),
    ("*ptr = x", "*ptr = x;", "STR R0, [R1]", "指针赋值"),
    ("x = arr[i]", "x = arr[i];", "LDR R0, [R1, R2, LSL #2]", "数组访问 (i*4+base)"),
    ("x = y << 2", "x = y << 2;", "LSL R0, R1, #2", "左移"),
    ("x = y & 0xFF", "x = y & 0xFF;", "AND R0, R1, #0xFF", "位掩码"),
    ("if x==y: ...", "if(x==y){...}", "CMP R0, R1  \n  BNE skip", "条件比较+跳转"),
    ("for i in range(n)", "for(i=0;i<n;i++)", "MOV R0,#0  \nloop: CMP R0,R2  \n  BGE done  \n  ADD R0,R0,#1  \n  B loop", "循环"),
    ("while x>0: ...", "while(x>0){...}", "loop: CMP R0,#0  \n  BLE done  \n  ...  \n  B loop", "while循环"),
]

for py, c, asm, note in examples:
    print(f"  {py:<23s} {c:<22s} {asm:<35s} {note}")

print("\n💡 你现在看到的所有 Python 代码，最终都会变成这些简单指令的序列")

---
## 🎯 综合挑战

In [ ]:
# Q1: 用汇编器写一个程序，计算两个数的平方差 a² - b²
# 提示：a² = a×a, b² = b×b, 然后相减

asm3 = Assembler()

diff_squares_code = """
    ; a=7 在 [0x90], b=4 在 [0x91]
    ; 计算 a² - b² = 49 - 16 = 33
    
    ; 你的代码：
    
    
"""

# machine_code = asm3.assemble(diff_squares_code)
# cpu3 = CPU()
# cpu3.load_machine_code(machine_code)
# cpu3.mem.write(0x90, 7)  # a
# cpu3.mem.write(0x91, 4)  # b
# cpu3.run()
# print(f"\n🔍 mem[0x80] = {cpu3.mem.read(0x80)}  (预期: 49-16 = 33)")
print("✏️ 写完后取消注释")

In [ ]:
# Q2: RISC 和 CISC 的核心区别是什么？
# A) RISC 指令更多
# B) CISC 功耗更低
# C) RISC 只有 LOAD/STORE 访存，CISC 任何指令都可能访存
# D) RISC 指令不等长

q2 = ""  # 👈 你的答案
print(f"Q2 答案: {q2}")

In [ ]:
# Q3: 写出以下 Python 代码对应的汇编（用我们的简单 ISA）
#
# total = 0
# for i in range(1, 6):    # i = 1,2,3,4,5
#     total = total + i
#
# 最终 total = 15

asm4 = Assembler()

for_loop_code = """
    ; 在这里写你的汇编实现
    
    
"""

# machine_code = asm4.assemble(for_loop_code)
# cpu4 = CPU()
# cpu4.load_machine_code(machine_code)
# cpu4.mem.write(0xA0, 0)   # total
# cpu4.mem.write(0xA1, 1)   # i = 1
# cpu4.mem.write(0xA2, 5)   # limit
# cpu4.mem.write(0xA3, 1)   # step
# cpu4.run()
# print(f"\n🔍 mem[0x80] = {cpu4.mem.read(0x80)}  (预期: 15)")
print("✏️ 写完后取消注释")

---
## ✅ 检查清单

对照一下，都掌握了吗？

- [ ] ISA 是什么：CPU 和软件的接口契约
- [ ] 指令格式：操作码 + 操作数
- [ ] CISC (x86) vs RISC (ARM) 的区别和各自优劣
- [ ] 三大类指令：数据处理、访存、控制流
- [ ] 寻址模式：立即数、寄存器、间接、基址偏移等
- [ ] 调用约定：参数/返回值/栈帧/callee-saved vs caller-saved
- [ ] 能写简单的汇编程序（循环、条件、算术）
- [ ] 理解高级语言如何映射到汇编

---

> 📖 下一章：[04 - 存储层次](../04-memory-hierarchy/) — 从寄存器到硬盘，数据跨越多层存储的旅程